In [1]:
%pip install pandas


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd

data_dir = Path("/Users/sahil/Name that LLM/Name-That-LLM/data/raw")

train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")

print("Train shape:", train.shape)  # (2699, 4)
print("Test shape:", test.shape)    # (3, 2)

Train shape: (2699, 4)
Test shape: (3, 2)


In [12]:
train.head(10)

model_counts = {}

for model in train["model"]:
    model_counts[model] = model_counts.get(model, 0) + 1

for model, count in model_counts.items():
    print(f"{model}: {count}")

train["model"].value_counts()
train["model"].nunique()
    

qwen3.8-max: 150
claude-sonnet-5: 150
gemini-3.6-flash: 150
gpt-5.6-terra: 150
glm-5.3-flash: 150
gemini-3.1-pro-preview: 150
gemini-3.8-flash: 150
claude-opus-4-6: 150
gpt-5.6-luna: 150
claude-opus-4-8: 150
gemini-3.7-flash: 150
claude-fable-5: 149
claude-opus-5: 150
kimi-k3: 150
gpt-5.5: 150
gpt-6-astra: 150
glm-5.3: 150
gpt-5.6-sol: 150


18

In [4]:
train.columns.to_list()


['id', 'prompt_id', 'text', 'model']

In [5]:
test.head(10)

,id,text
0,00353fb1c541,"The more efficiently we use a resource, the mo..."
1,005f5a9a6139,"The Stern-Gerlach experiment, performed in 192..."
2,0062a53d5487,"Hundreds of millions of years ago, fluctuating..."


In [6]:
test.columns.to_list()

['id', 'text']

In [13]:
#Building my own Transformer from now on

In [16]:
#Build a Tokenizer
#Creating Vocabulary

import re
from collections import Counter

#Splitting tokens into words and punctuation

def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text)

#Count tokens across all training texts

token_counts = Counter()

for text in train["text"]:
    token_counts.update(tokenize(text))

#Reserve IDs for special tokens.
vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
}

#Assign a unique ID to every observed token.
for token, count in token_counts.most_common():
    vocab[token] = len(vocab)

print("Vocabulary size", len(vocab))
print(list(vocab.items())[:20])

Vocabulary size 19130
[('<PAD>', 0), ('<UNK>', 1), (',', 2), ('the', 3), ('.', 4), ('a', 5), ('and', 6), ('of', 7), ('to', 8), ('that', 9), ('in', 10), ('is', 11), ('into', 12), ('it', 13), ('s', 14), ('—', 15), ('not', 16), (':', 17), ('its', 18), ('as', 19)]


In [17]:
text = "Hello, world!"

tokens = tokenize(text)
token_ids = [vocab.get(token, vocab["<UNK>"]) for token in tokens]

print(tokens)
print(token_ids)

['Hello', ',', 'world', '!']
[1, 2, 60, 4136]


In [18]:
#Creating the Embedding Layer

import random

V = len(vocab)
d = 8

rng = random.Random(42)

E = [
    [ rng.uniform(-0.1, 0.1) for _ in range(d)]
    for _ in range(V)
]

#Set the padding token's embedding to zeros.
E[vocab["<PAD>"]] = [0.0] * d

print("Rows:", len(E)) #V
print("Columns:", len(E[0])) #8

Rows: 19130
Columns: 8


In [25]:
#Retrieving Token Embedding:
token_id = vocab.get("Hello", vocab["<UNK>"])
e_i = E[token_id]

print(e_i)

[-0.015615636062945915, -0.09404055611238593, -0.05627240503927933, 0.001071057620672472, -0.09469280606322728, -0.06023246986267031, 0.029976887555904635, 0.008988296120643335]


In [26]:
#Impleting position vector
import math

def position_vector(position, d):
    if d<=0 or d % 2 != 0:
        raise ValueError("d must be positive even number")

    vector = []

    for dimension in range(0,d,2):
        angle = position / (10000 ** (dimension/d))

        vector.append(math.sin(angle))
        vector.append(math.cos(angle))

    return vector


In [29]:
print(position_vector(1,6))

[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]


In [30]:
import math


def self_attention(X, W_Q, W_K, W_V):
    """
    Shapes:
        X:   sequence_length × d_model
        W_Q: d_model × d_k
        W_K: d_model × d_k
        W_V: d_model × d_v

    Returns:
        output:  sequence_length × d_v
        weights: sequence_length × sequence_length
    """
    if not X:
        return [], []

    def matmul(A, B):
        columns = list(zip(*B))

        return [
            [
                sum(a * b for a, b in zip(row, column))
                for column in columns
            ]
            for row in A
        ]

    # 1. Project the same input into queries, keys, and values.
    Q = matmul(X, W_Q)
    K = matmul(X, W_K)
    V = matmul(X, W_V)

    # 2. Compute scaled similarity scores: QKᵀ / sqrt(d_k).
    d_k = len(Q[0])
    K_transposed = list(zip(*K))
    scores = matmul(Q, K_transposed)

    scores = [
        [score / math.sqrt(d_k) for score in row]
        for row in scores
    ]

    # 3. Apply softmax separately to each query's scores.
    weights = []

    for row in scores:
        maximum = max(row)
        exponentials = [math.exp(score - maximum) for score in row]
        total = sum(exponentials)

        weights.append([value / total for value in exponentials])

    # 4. Produce a weighted combination of the value vectors.
    output = matmul(weights, V)

    return output, weights

In [31]:
import random

from name_that_llm.tokenizer import encode

# 1. Convert text into token IDs.
token_ids = encode("Hello, world!", vocab)

d_model = len(E[0])

# 2. Add positional information to each token embedding.
X = []

for position, token_id in enumerate(token_ids):
    embedding = E[token_id]
    positional = position_vector(position, d_model)

    X.append([
        embedding[j] + positional[j]
        for j in range(d_model)
    ])

# 3. Initialize the projection matrices.
rng = random.Random(42)

d_k = d_model
d_v = d_model

def random_matrix(rows, columns):
    return [
        [rng.uniform(-0.1, 0.1) for _ in range(columns)]
        for _ in range(rows)
    ]

W_Q = random_matrix(d_model, d_k)
W_K = random_matrix(d_model, d_k)
W_V = random_matrix(d_model, d_v)

# 4. Execute self-attention.
output, weights = self_attention(X, W_Q, W_K, W_V)

# 5. Inspect the results.
print("Token IDs:", token_ids)
print("Output shape:", len(output), "×", len(output[0]))
print("Attention shape:", len(weights), "×", len(weights[0]))

print("\nAttention weights:")
for row in weights:
    print([round(value, 4) for value in row])

Token IDs: [1, 2, 60, 4136]
Output shape: 4 × 8
Attention shape: 4 × 4

Attention weights:
[0.2493, 0.2508, 0.2511, 0.2488]
[0.2508, 0.2512, 0.25, 0.248]
[0.252, 0.2518, 0.2494, 0.2468]
[0.2517, 0.2518, 0.2497, 0.2468]


In [32]:
def residual_add(X, attention_output):
    if len(X) != len(attention_output):
        raise ValueError("Inputs must have the same number of rows")

    result = []

    for input_row, attention_row in zip(X, attention_output):
        if len(input_row) != len(attention_row):
            raise ValueError("Corresponding rows must have the same length")

        result.append([
            x + a
            for x, a in zip(input_row, attention_row)
        ])

    return result